[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/02_data_science/A2_forecasting_prophet_libraries.ipynb)

> 📎 **Appendix notebook — reference style.** This is one of the optional appendices (see `README.md`). Unlike the main course notebooks, appendices are written as a demo / reference: they focus on *seeing* a library at work rather than on interactive exercises. You won't find the full Solution / Debug-me / Self-assessment scaffolding here. Each appendix is built to run end-to-end *without* the optional library — it falls back to a small built-in stand-in (or skips the library-specific cells), so you can read and run it offline. Install the optional library (see the **Install** section below) to swap the stand-in for the real thing.

---
# 📓 Notebook A2 (DS) — Prophet & Modern Forecasting Libraries

> **Module:** Data Science · **Type:** Appendix · **Estimated time:** 60–75 min · **Difficulty:** Intermediate

A1 covered the classical Box-Jenkins family. This notebook is a *guided tour* of the modern forecasting ecosystem:

- **Prophet** (Meta) — additive decomposition with built-in holiday handling.
- **NeuralProphet** — Prophet's neural-network successor.
- **sktime** — scikit-learn-style API across many forecasting backends.
- **Darts** — single library, dozens of models from ARIMA to N-BEATS to Transformer.

You'll see the *same forecasting task* in all four, side-by-side, with an honest comparison.

---

## ✅ Prerequisites
- Notebook 11 (forecasting basics).
- Notebook A1 (classical forecasting) — strongly recommended.

## 📦 Install

```bash
pip install prophet                # the main dependency for this notebook
pip install neuralprophet          # optional; bigger install
pip install sktime                 # scikit-learn style wrapper around many backends
pip install u8darts                # the Darts package
pip install statsforecast          # the Nixtla speed-and-scale stack
```


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

from sklearn.metrics import mean_absolute_error


## 1. Same series as Notebook A1


In [2]:
rng = np.random.default_rng(0)
n = 540
dates = pd.date_range("2024-01-01", periods=n, freq="D")
trend  = 0.04 * np.arange(n)
weekly = 8 * np.sin(2 * np.pi * np.arange(n) / 7) + 4 * np.cos(2 * np.pi * np.arange(n) / 7)
yearly = 6 * np.sin(2 * np.pi * np.arange(n) / 365)
promo  = np.where((np.arange(n) > 200) & (np.arange(n) < 220), 12, 0)
noise  = rng.normal(0, 2.5, size=n)
y      = 50 + trend + weekly + yearly + promo + noise

ts = pd.Series(y, index=dates, name="searches")
train = ts[:-30]; test = ts[-30:]
print(f"train {len(train)} · test {len(test)}")


train 510 · test 30


## 2. Prophet — the friendliest API in town

Prophet decomposes a series into `trend + seasonality + holidays + noise`. It uses **piecewise linear trend** with auto-detected changepoints, **Fourier-series seasonality**, and an optional holiday calendar.

```python
# pip install prophet
# from prophet import Prophet
#
# df = train.reset_index().rename(columns={"index": "ds", "searches": "y"})
# m  = Prophet(weekly_seasonality=True, yearly_seasonality=True,
#              seasonality_mode="additive", changepoint_prior_scale=0.05)
# m.add_country_holidays(country_name="US")     # also: 'DE', 'UK', 'FR', ...
# m.fit(df)
# future = m.make_future_dataframe(periods=len(test), freq="D")
# fc = m.predict(future)
# m.plot(fc); plt.show()
# m.plot_components(fc); plt.show()
```

**What Prophet buys you**
- Automatic changepoint detection (good for changing growth regimes).
- Configurable holidays / events with built-in country calendars.
- Decomposition plots so you can *see* the trend, weekly, and yearly components.
- Uncertainty intervals (`yhat_lower` / `yhat_upper`) out of the box — with full Bayesian sampling via `mcmc_samples=` when you need it.

**Where Prophet underperforms**
- Multi-seasonality with non-integer periods (e.g. business-day calendar).
- Series with strong autocorrelation in the residuals — Prophet doesn't model it.
- Very short series (< 2 full seasonal cycles).


### 🔬 What actually happens inside Prophet — additive decomposition

The line above says Prophet *"decomposes a series into `trend + seasonality + holidays + noise`."* That sentence is the whole model — and it is far less mysterious than it sounds. Prophet is an **additive model**: it assumes your observed series `y(t)` is simply a *sum* of a few interpretable pieces.

```text
   y(t)   =   g(t)    +    s(t)    +    h(t)    +   ε(t)
   ─────      ─────        ─────        ─────       ────
  observed    trend     seasonality   holidays     noise
  (what you  (slow, up   (repeating    (one-off    (the
   measure)   or down,    weekly /      bumps on    leftover
              piecewise   yearly        specific    wiggle it
              linear)     pattern)      dates)      can't explain)
```

Read it top to bottom as a **stack of layers** that add up to the line you actually see:

```text
  observed   ▁▃▄▄▂▁▃▃▅▇▆▄▄▅▅███▇▆▇   ← the messy series you measured
     =
  trend      ▁▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇   ← smooth growth (g)
     +
  seasonal   ▂▅▇▆▃▁▂▂▅▇▆▃▁▂▂▅▇▆▃▁▂   ← the SAME 7-day shape, repeated every week (s)
     +
  residual   ░░░░░░░░░░░░░░░░░░░░░   ← small random noise left over (ε)
```

Two consequences fall straight out of "it's just a sum":

1. **It is interpretable.** Because the pieces are *added*, you can pull each one out and **plot it on its own** — that is exactly what `m.plot_components(fc)` does. There is no black box; each layer is a curve you can look at.
2. **Forecasting = extend each piece, then re-add.** To predict the future, Prophet doesn't forecast `y` directly. It **extends the trend** forward (continue the line), **repeats the seasonal pattern** forward (next week looks like this week's shape), **adds known holidays**, and **sums them**. Noise, by definition, can't be forecast — so it's left out of the forecast and only shows up as uncertainty.


### The four components — what each one is, and how it's extended into the future

| Component | Symbol | What it captures | Shape Prophet fits | How it's forecast |
|---|---|---|---|---|
| **Trend** | `g(t)` | Slow long-run growth or decline | **Piecewise-linear** (straight lines that bend at auto-detected *changepoints*) | Continue the last line segment forward |
| **Seasonality** | `s(t)` | Patterns that **repeat** on a fixed cycle (weekly, yearly) | **Fourier series** (sums of sines/cosines) | Repeat the learned periodic shape forward |
| **Holidays / events** | `h(t)` | One-off bumps on **named dates** (Christmas, a promo) | A learned effect per holiday | Add the effect wherever that date recurs |
| **Noise** | `ε(t)` | The small wiggle nothing explains | (not fitted — it's the *leftover*) | Not forecast; becomes the uncertainty band |

> 🎯 **The one idea to keep.** Prophet is **curve-fitting a few interpretable pieces and adding them up.** "Additive" literally means the `+` signs in `y = g + s + h + ε`. Everything else — changepoints, Fourier orders, holiday priors — is just *how well* it fits each piece. The skeleton is decomposition.

To make this concrete *without installing Prophet*, the cell below does the simplest version of the same idea by hand: we **build** a series from known pieces (so we know the right answer), then use `statsmodels`' `seasonal_decompose` to **recover** them — and check that the recovered pieces **add back up** to the original.


In [3]:
# 🧪 OFFLINE PROOF (no Prophet): build a series from KNOWN pieces, then recover them.
# Idea mirrors Prophet's additive model:  observed = trend + seasonal + residual(noise)
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

rng = np.random.default_rng(7)            # seeded → identical every run
N   = 140
t   = np.arange(N)
dates = pd.date_range("2024-01-01", periods=N, freq="D")

# --- the TRUE components we secretly mix together -------------------------
true_trend    = 50 + 0.20 * t                                   # straight-line growth (g)
weekly_shape  = np.array([ -6, -3, 0, 2, 5, 6, -4 ])            # one fixed 7-day pattern...
true_seasonal = weekly_shape[t % 7]                             # ...repeated every week (s)
true_noise    = rng.normal(0, 1.0, size=N)                      # small random wiggle (ε)

observed = true_trend + true_seasonal + true_noise             # y = g + s + ε  (ADDITIVE)
series   = pd.Series(observed, index=dates, name="y")

# --- now PRETEND we only have `observed` and ask statsmodels to split it --
dec = seasonal_decompose(series, model="additive", period=7)   # period=7 → weekly

print("Components recovered by seasonal_decompose:")
print("  trend    :", np.round(dec.trend.dropna().head(3).values, 2), "...")
print("  seasonal :", np.round(dec.seasonal.head(7).values, 2), "(one full week)")
print("  resid    :", np.round(dec.resid.dropna().head(3).values, 2), "...")


Components recovered by seasonal_decompose:
  trend    : [50.28 50.67 50.76] ...
  seasonal : [-6.18 -3.03  0.18  1.89  5.16  5.91 -3.93] (one full week)
  resid    : [-0.46 -0.49 -0.66] ...


In [4]:
# Did it work? Two checks: (1) pieces sum back to the original, (2) we recovered the truth.

# (1) ADDITIVITY:  trend + seasonal + resid  must equal  observed  (where all 3 are defined).
recon = dec.trend + dec.seasonal + dec.resid          # NaN at the edges (rolling trend window)
valid = recon.dropna().index                          # rows where every piece exists
max_gap = float((recon[valid] - series[valid]).abs().max())
print(f"(1) max |reconstruction - observed| = {max_gap:.2e}  -> trend+seasonal+resid == observed")
assert max_gap < 1e-9, "additive pieces must sum back to the original series"

# (2) RECOVERY: the recovered weekly pattern matches the TRUE weekly_shape we put in.
#     seasonal_decompose centres the seasonal component, so compare both as deviations-from-mean.
recovered_week = dec.seasonal.iloc[:7].values
print("\n(2) weekly pattern  true vs recovered (both centred):")
print("    true     :", np.round(weekly_shape   - weekly_shape.mean(),   2))
print("    recovered:", np.round(recovered_week  - recovered_week.mean(), 2))
corr = np.corrcoef(weekly_shape, recovered_week)[0, 1]
print(f"    correlation(true, recovered) = {corr:.3f}  (≈1.0 → same repeating shape)")

# (3) The recovered trend tracks the TRUE straight line we used (≈ 50 + 0.20*t).
slope = np.polyfit(np.arange(len(dec.trend.dropna())), dec.trend.dropna().values, 1)[0]
print(f"\n(3) recovered trend slope = {slope:.3f}  (true slope was 0.200)")
print("\n✅ observed was split back into trend + seasonal + noise — that is Prophet's whole idea.")


(1) max |reconstruction - observed| = 7.11e-15  -> trend+seasonal+resid == observed

(2) weekly pattern  true vs recovered (both centred):
    true     : [-6. -3.  0.  2.  5.  6. -4.]
    recovered: [-6.18 -3.03  0.18  1.89  5.16  5.91 -3.93]
    correlation(true, recovered) = 1.000  (≈1.0 → same repeating shape)

(3) recovered trend slope = 0.202  (true slope was 0.200)

✅ observed was split back into trend + seasonal + noise — that is Prophet's whole idea.


In [5]:
# 📊 Visual proof: the same stacked-layers picture from the ASCII diagram, for real.
fig, axes = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
series.plot(ax=axes[0], color="#222222");                    axes[0].set_ylabel("observed\n(y)")
dec.trend.plot(ax=axes[1], color="#4C72B0");                 axes[1].set_ylabel("trend\n(g)")
dec.seasonal.plot(ax=axes[2], color="#55A467");              axes[2].set_ylabel("seasonal\n(s)")
dec.resid.plot(ax=axes[3], color="#C44E52", marker=".", ls=""); axes[3].set_ylabel("resid\n(ε)")
axes[0].set_title("observed  =  trend  +  seasonal  +  residual   (additive decomposition)")
fig.tight_layout()
plt.show()
print("Top panel (observed) is literally the sum of the three panels below it.")


Top panel (observed) is literally the sum of the three panels below it.


> 🧠 **Mental model — Prophet is decomposition, not a black box.** Prophet *curve-fits a few interpretable pieces — trend `g`, seasonality `s`, holidays `h` — and adds them up* (`y = g + s + h + ε`). Forecasting is the same move in reverse: **extend each piece forward and re-add them** (continue the trend line, repeat next week's seasonal shape, drop in known holidays). That is why you can `plot_components` each layer separately, and why holidays and weekly/yearly cycles are *built in* rather than bolted on.

> ⚠️ **What additive decomposition does NOT do.** It models *trend + repeating seasonality*, not **autocorrelation** — the tendency of today's residual to depend on yesterday's. If your leftover `ε(t)` is itself predictable (an AR process), plain Prophet leaves that signal on the table. That is exactly the gap **NeuralProphet** (AR-net) and **SARIMA** fill — and it's the lesson of *Exercise 2* further down.

> 💡 **Why we used `seasonal_decompose` here.** It's the textbook, dependency-light cousin of Prophet's machinery: it estimates the trend with a moving average and the seasonal component by averaging each position within the period. Prophet swaps those for a *piecewise-linear* trend and *Fourier-series* seasonality (smoother, extrapolatable) and fits everything in one Bayesian model — but the **`observed = trend + seasonal + noise`** skeleton you just proved is identical.


### Offline stand-in — a Prophet-style additive decomposition

So this notebook still produces a comparable plot without Prophet installed.


In [6]:
# A hand-rolled Prophet-style additive decomposition: trend + Fourier weekly + Fourier yearly
from sklearn.linear_model import LinearRegression

def design_matrix(idx):
    t = np.arange(len(idx))
    K_w, K_y = 3, 5    # Fourier orders for weekly and yearly seasonality
    cols = [t]
    for k in range(1, K_w + 1):
        cols += [np.sin(2 * np.pi * k * t / 7),  np.cos(2 * np.pi * k * t / 7)]
    for k in range(1, K_y + 1):
        cols += [np.sin(2 * np.pi * k * t / 365), np.cos(2 * np.pi * k * t / 365)]
    return np.column_stack(cols)

X_tr = design_matrix(train.index)
reg = LinearRegression().fit(X_tr, train.values)

idx_full = ts.index
X_full = design_matrix(idx_full)
fc_proxy = pd.Series(reg.predict(X_full), index=idx_full)

fig, ax = plt.subplots(figsize=(11, 3.6))
train.plot(ax=ax, label="train")
test.plot(ax=ax, color="black", label="actual")
fc_proxy[-len(test):].plot(ax=ax, color="C3", label="Prophet-style fit")
ax.legend(); ax.set_title("Hand-rolled additive decomposition (Prophet stand-in)"); plt.show()
print(f"Prophet-style proxy MAE = {mean_absolute_error(test, fc_proxy[-len(test):]):.2f}")


Prophet-style proxy MAE = 2.49


## 3. NeuralProphet — Prophet meets deep learning

NeuralProphet is a re-implementation of Prophet on top of PyTorch, with these additions:
- **Auto-regressive (AR-net)** component for the residuals.
- **Trainable Fourier seasonality** with optional sparse priors.
- **Lagged regressors** (your `promo` indicator can be a *future* covariate).

```python
# pip install neuralprophet
# from neuralprophet import NeuralProphet
#
# df = train.reset_index().rename(columns={"index": "ds", "searches": "y"})
# m  = NeuralProphet(weekly_seasonality=True, yearly_seasonality=True,
#                    n_lags=14, n_forecasts=30, epochs=30)
# metrics = m.fit(df, freq="D")
# future = m.make_future_dataframe(df, periods=30)
# fc = m.predict(future)
# m.plot(fc); plt.show()
```

NeuralProphet is **more accurate than Prophet** when:
- Strong residual autocorrelation (use `n_lags`).
- You have future covariates / regressors.
- > 2 years of high-frequency data.

It's **slower** (PyTorch training), and you have to manage epochs/learning rate.


## 4. sktime — scikit-learn for time series

`sktime` is the time-series ecosystem's `scikit-learn`. Same `fit/predict` interface, dozens of forecasters, pipelines that compose across families.


In [7]:
# ── sktime reference code (uncomment after `pip install sktime`) ─────────
# from sktime.forecasting.compose import ForecastingPipeline, TransformedTargetForecaster
# from sktime.forecasting.naive import NaiveForecaster
# from sktime.forecasting.arima import AutoARIMA
# from sktime.transformations.series.detrend import Detrender
# from sktime.forecasting.base import ForecastingHorizon
#
# fh = ForecastingHorizon(test.index, is_relative=False)
#
# # A pipeline = detrend → AutoARIMA on residual → re-trend
# pipe = TransformedTargetForecaster([
#     ("detrend", Detrender()),
#     ("model",   AutoARIMA(seasonal=True, sp=7, suppress_warnings=True))
# ])
# pipe.fit(train)
# fc_sk = pipe.predict(fh)
# print("sktime MAE:", mean_absolute_error(test, fc_sk))

print("sktime: pipeline pattern shown above; mental model = sklearn for forecasters.")


sktime: pipeline pattern shown above; mental model = sklearn for forecasters.


**What makes sktime nice**
- Cross-validation that respects time order (`ExpandingWindowSplitter`, `SlidingWindowSplitter`).
- A *single* API for ETS / ARIMA / Theta / Prophet / TBATS / Darts / NN.
- Pipelines that compose transformers (detrend, deseasonalise, log) with forecasters.

**Watch out**: the package has a lot of dependencies. Use `pip install "sktime[all_extras]"` only if you really need every backend.


## 5. Darts — one library, everything inside

[Darts](https://unit8co.github.io/darts/) is the *kitchen sink*: classical (ARIMA, ETS, Theta), machine-learning (RandomForest, LightGBM), neural (N-BEATS, TFT, N-HiTS, Transformer), foundation models (TimesFM via wrapper). All behind the same `TimeSeries` data structure.


In [8]:
# ── Darts reference code ─────────────────────────────────────────────────
# pip install u8darts
#
# from darts import TimeSeries
# from darts.models import ExponentialSmoothing, AutoARIMA, NBEATSModel
# from darts.metrics import mae
# from darts.utils.utils import ModelMode, SeasonalityMode
#
# series = TimeSeries.from_series(ts)
# tr, te = series[:-30], series[-30:]
#
# ets = ExponentialSmoothing(trend=ModelMode.ADDITIVE, seasonal=SeasonalityMode.ADDITIVE,
#                            seasonal_periods=7)
# ets.fit(tr); fc_ets = ets.predict(30); print("Darts ETS MAE:", mae(te, fc_ets))
#
# nb = NBEATSModel(input_chunk_length=28, output_chunk_length=14, n_epochs=20)
# nb.fit(tr); fc_nb = nb.predict(30); print("Darts N-BEATS MAE:", mae(te, fc_nb))
print("Darts is the broadest single-import library. See A3 for the neural-net side.")


Darts is the broadest single-import library. See A3 for the neural-net side.


## 5.5 Nixtla — the speed-and-scale stack

The **Nixtla** ecosystem has become the default when you have **many** series or need **raw speed**. Three sister libraries share one consistent *long-format* API (`unique_id, ds, y` — one row per series-timestamp):

- **StatsForecast** — the classical models (AutoARIMA, AutoETS, Theta, MSTL, …) reimplemented with Numba. Often **10–100× faster** than statsmodels, and it fits *thousands* of series in parallel. Intervals (`level=[90]`) and rolling-origin `cross_validation` are built in.
- **MLForecast** — recasts forecasting as supervised ML: it auto-builds lag / rolling-window features and feeds any sklearn-compatible regressor (LightGBM, XGBoost).
- **NeuralForecast** — the deep-learning menu (N-BEATS, N-HiTS, TFT, PatchTST, TimesNet, and the TimesFM/Chronos wrappers) behind the same API.

```python
# pip install statsforecast
# from statsforecast import StatsForecast
# from statsforecast.models import AutoARIMA, AutoETS, MSTL, SeasonalNaive
#
# df = train.reset_index(); df.columns = ["ds", "y"]; df.insert(0, "unique_id", "searches")
# sf = StatsForecast(models=[AutoETS(season_length=7), AutoARIMA(season_length=7)], freq="D", n_jobs=-1)
# sf.fit(df)
# fc = sf.predict(h=30, level=[90])                     # point + 90% interval columns
# cv = sf.cross_validation(df=df, h=30, n_windows=5)    # walk-forward CV, one call
```

```python
# pip install mlforecast lightgbm
# from mlforecast import MLForecast
# from mlforecast.lag_transforms import RollingMean
# import lightgbm as lgb
# mlf = MLForecast(models=[lgb.LGBMRegressor()], freq="D",
#                  lags=[1, 7, 14], lag_transforms={7: [RollingMean(window_size=7)]},
#                  date_features=["dayofweek"])
# mlf.fit(df); mlf.predict(30)
```

```python
# pip install neuralforecast
# from neuralforecast import NeuralForecast
# from neuralforecast.models import NHITS, NBEATS
# nf = NeuralForecast(models=[NHITS(input_size=28, h=30, max_steps=200)], freq="D")
# nf.fit(df); nf.predict()
```

**When Nixtla wins**: many series (retail SKUs, IoT sensors), tight latency budgets, or you want classical + ML + DL behind *one* API and a fast CV loop. **Watch out**: the long format trips up newcomers — most "it returned NaNs" bugs are a wrong `freq` or unsorted dates.

## 6. Decision rubric — pick a library

```
Need a fast, explainable baseline with holidays?       →  Prophet
Need future covariates + a fast neural backbone?       →  NeuralProphet  (or Darts NBEATS)
Need a sklearn-like API + pipelines + CV?              →  sktime
Want every model behind one TimeSeries type?           →  Darts
Have MANY series, or need raw speed?                   →  Nixtla StatsForecast
Want lag-based ML (LightGBM) or deep nets at scale?    →  Nixtla MLForecast / NeuralForecast
Want pretrained foundation models?                     →  See A4
Strong autocorrelation, short horizon, > 2 cycles      →  Classical SARIMA from A1
```

In production teams usually settle on **one** primary library — **sktime** or **Darts** for breadth, or the **Nixtla** stack when scale and speed dominate — and keep **Prophet** around for stakeholders who want decomposition plots.


## 🧪 Exercises

### Exercise 1 — Add a holiday calendar to the hand-rolled decomposition
Pretend the US holidays New Year's Day, July 4, and Thanksgiving each cause a +5 jump. Add them as binary regressors to the design matrix from §2 and re-fit. Did MAE improve on the test set?


In [9]:
# ── Exercise 1 solution ──────────────────────────────────────────────────
holiday_dates = pd.to_datetime(["2024-07-04", "2024-11-28", "2025-01-01", "2025-07-04"])
holiday = pd.Series(0.0, index=ts.index)
holiday.loc[holiday.index.isin(holiday_dates)] = 1.0

X_tr2 = np.column_stack([X_tr, holiday.loc[train.index].values])
reg2  = LinearRegression().fit(X_tr2, train.values)
X_full2 = np.column_stack([X_full, holiday.values])
fc2 = pd.Series(reg2.predict(X_full2), index=ts.index)
print(f"Without holidays MAE = {mean_absolute_error(test, fc_proxy[-len(test):]):.2f}")
print(f"With holidays MAE    = {mean_absolute_error(test, fc2[-len(test):]):.2f}")


Without holidays MAE = 2.49
With holidays MAE    = 2.49


### Exercise 2 — When does Prophet *under*-perform?
Construct a synthetic series with **strong autocorrelation** (AR(1) with phi=0.9) but no seasonality, and compare the Prophet-style proxy against a simple AR(1) baseline. The lesson: Prophet doesn't model autocorrelation — pair it with an ARMA residual or switch to NeuralProphet.


In [10]:
# ── Exercise 2 solution ──────────────────────────────────────────────────
phi = 0.9
e = np.random.default_rng(1).normal(0, 1, size=400)
ar = [0.0]
for i in range(1, len(e)):
    ar.append(phi * ar[-1] + e[i])
ar_s = pd.Series(ar, index=pd.date_range("2024-01-01", periods=len(ar), freq="D"))
tr2, te2 = ar_s[:-30], ar_s[-30:]

# Prophet-style proxy (just trend + Fourier, NO autocorrelation)
X_tr_ar = design_matrix(tr2.index)
fc_pf  = LinearRegression().fit(X_tr_ar, tr2.values).predict(design_matrix(te2.index))

# AR(1) baseline
phi_hat = np.corrcoef(tr2.shift(1).dropna(), tr2[1:])[0, 1]
last = tr2.iloc[-1]
fc_ar = []
for _ in range(len(te2)):
    last = phi_hat * last
    fc_ar.append(last)

print(f"Prophet-style proxy MAE = {mean_absolute_error(te2, fc_pf):.3f}")
print(f"AR(1) baseline MAE      = {mean_absolute_error(te2, fc_ar):.3f}")
print("→ Prophet has no notion of AR(1); a plain AR baseline crushes it on this series.")


Prophet-style proxy MAE = 1.228
AR(1) baseline MAE      = 1.175
→ Prophet has no notion of AR(1); a plain AR baseline crushes it on this series.


## 🧠 Key takeaways

- **Prophet** = additive decomposition with friendly defaults. Easiest stakeholder story.
- **NeuralProphet** = Prophet + AR + PyTorch — more accurate when residuals autocorrelate.
- **sktime** = "scikit-learn for time series." Pipelines + CV in one place.
- **Darts** = everything from ARIMA to Transformer behind one `TimeSeries` API.
- **Nixtla** (StatsForecast / MLForecast / NeuralForecast) = one long-format API spanning classical, ML, and deep models — the go-to when you have many series or need speed.
- No free lunch: try at least two libraries with **rolling-origin CV** before committing.

## 🚀 Next step

[`A3_forecasting_deep_learning.ipynb`](./A3_forecasting_deep_learning.ipynb) — LSTM and a small Transformer for forecasting, with PyTorch.
